## **Olist E-commerce Exploratory Data Analysis**

## **About Data**

Raw transactional data is often messy and full of duplicates. If orders, payments, and items are merged directly into one table, a single order with multiple items or split payments can inflate sales numbers and distort the analysis.

To avoid this, the data is organized using a structured star-schema approach. Instead of one large table it is split into three separate tables each focused on a specific level of detail:

Orders table (df_orders): Each row represents one order, giving a clear overview of overall business performance, delivery times, and customer satisfaction without duplication
Products analysis table (df_products_analysis): Each row represents a single product item within an order, allowing detailed tracking of products, categories, and sellers
Payments analysis table (df_payments_analysis): Each row represents a single payment transaction, helping analyze payment methods, installments, and customer spending behavior


## **Importing Libraries** 

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')
pio.renderers.default="iframe_connected"

#----------------------------------------------------------------------------#
custom_dark = dict(
    layout=dict(
        paper_bgcolor="#161625",  
        plot_bgcolor="#161625",    

        font=dict(color="#E0E0E0"),  

        xaxis=dict(
            gridcolor="#212135",
            zerolinecolor="#2C2C44",
            color="#E0E0E0"
        ),

        yaxis=dict(
            gridcolor="#212135",
            zerolinecolor="#2C2C44",
            color="#E0E0E0"
        ),

        colorway=px.colors.qualitative.Dark24,  
    )
)
#-----------------------------------------------------------------------------#

pio.templates["custom_dark"] = custom_dark
pio.templates.default = "custom_dark"

In [2]:
olist_orders_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv')
olist_order_customer_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv') 
olist_order_payments_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv')
olist_order_items_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv')
olist_products_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv')
product_category_name_translation = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv')
olist_sellers_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv')
olist_order_reviews_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv')
olist_geolocation_dataset = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv')


# Pre Aggregating 
# 1. Summarizing payments per order
pay_summary = olist_order_payments_dataset.groupby('order_id')['payment_value'].sum().reset_index(name='total_paid')

# 2. Summarizing items per order
items_summary = olist_order_items_dataset.groupby('order_id').agg(
    total_items=('order_item_id', 'max'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum')
).reset_index()

# 3. Summarizing reviews per order One order can have multiple reviews we take the average score
review_summary = olist_order_reviews_dataset.groupby('order_id')['review_score'].mean().reset_index(name='avg_review_score')

# 4. Cleaning Geolocation Taking average lat and lng for each zip code prefix so it doesn't crash
geo_summary = olist_geolocation_dataset.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

# MASTER TABLE 1: Orders Analysis 
# This table is perfect for overall sales, delivery times, customer maps and reviews
# Connect orders with customer info
df_orders = olist_orders_dataset.merge(olist_order_customer_dataset, on='customer_id', how='left')

# Add the clean summaries Payments, Items and Reviews
df_orders = df_orders.merge(pay_summary, on='order_id', how='left')
df_orders = df_orders.merge(items_summary, on='order_id', how='left')
df_orders = df_orders.merge(review_summary, on='order_id', how='left')

# Add customer geolocation coordinates using their zip code
df_orders = df_orders.merge(geo_summary, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df_orders.drop(columns=['geolocation_zip_code_prefix'], inplace=True)



# MASTER TABLE 2: Payments Analysis 
# Special table just to check credit card vs cash trends without ruining sales totals

df_payments_analysis = olist_order_payments_dataset.merge(
    olist_orders_dataset[['order_id', 'order_purchase_timestamp', 'customer_id']], 
    on='order_id', 
    how='left'
)
df_payments_analysis = df_payments_analysis.merge(
    olist_order_customer_dataset[['customer_id', 'customer_state', 'customer_city']], 
    on='customer_id', 
    how='left'
)


# MASTER TABLE 3: Products and Sellers Analysis 
# This table connects the item sold with BOTH its product details and the seller details

# 1. Translate product categories to English
products_translated = olist_products_dataset.merge(
    product_category_name_translation, 
    on='product_category_name', 
    how='left'
)

# 2. Merge items with product details and seller details together
df_products_analysis = olist_order_items_dataset.merge(products_translated, on='product_id', how='left')
df_products_analysis = df_products_analysis.merge(olist_sellers_dataset, on='seller_id', how='left')

# 3. Add order details like date and status
df_products_analysis = df_products_analysis.merge(
    olist_orders_dataset[['order_id', 'order_purchase_timestamp', 'order_status']], 
    on='order_id', 
    how='left'
)


## **Orders Analysis**

In [3]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_paid,total_items,total_price,total_freight,avg_review_score,lat,lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,29.99,8.72,4.0,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,118.70,22.76,4.0,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,1.0,159.90,19.22,5.0,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,45.00,27.20,5.0,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,19.90,8.72,5.0,-23.676370,-46.514627


In [4]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99441 non-null  object 
 1   customer_id                    99441 non-null  object 
 2   order_status                   99441 non-null  object 
 3   order_purchase_timestamp       99441 non-null  object 
 4   order_approved_at              99281 non-null  object 
 5   order_delivered_carrier_date   97658 non-null  object 
 6   order_delivered_customer_date  96476 non-null  object 
 7   order_estimated_delivery_date  99441 non-null  object 
 8   customer_unique_id             99441 non-null  object 
 9   customer_zip_code_prefix       99441 non-null  int64  
 10  customer_city                  99441 non-null  object 
 11  customer_state                 99441 non-null  object 
 12  total_paid                     99440 non-null 

In [5]:
df_orders.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_zip_code_prefix,99441.0,35137.474583,29797.938996,1003.000000,11347.000000,24416.000000,58900.000000,99990.000000
total_paid,99440.0,160.990267,221.951257,0.000000,62.010000,105.290000,176.970000,13664.080000
total_items,98666.0,1.141731,0.538452,1.000000,1.000000,1.000000,1.000000,21.000000
total_price,98666.0,137.754076,210.645145,0.850000,45.900000,86.900000,149.900000,13440.000000
total_freight,98666.0,22.823562,21.650909,0.000000,13.850000,17.170000,24.040000,1794.960000
avg_review_score,98673.0,4.086793,1.346274,1.000000,4.000000,5.000000,5.000000,5.000000
lat,99163.0,-21.191224,5.608637,-33.689948,-23.589378,-22.924970,-20.139828,42.184003
lng,99163.0,-46.175442,4.056067,-72.668881,-48.097950,-46.630647,-43.598897,-8.723762


In [6]:
df_orders.describe(include="O").T

,count,unique,top,freq
order_id,99441,99441,66dea50a8b16d9b4dee7af250b4be1a5,1
customer_id,99441,99441,edb027a75a1449115f6b43211ae02a24,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-08-02 12:06:07,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-14 20:02:44,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_city,99441,4119,sao paulo,15540


### **Missing Values Handling**

In [7]:
df_orders.isna().sum()/len(df_orders)*100


order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
customer_unique_id               0.000000
customer_zip_code_prefix         0.000000
customer_city                    0.000000
customer_state                   0.000000
total_paid                       0.001006
total_items                      0.779357
total_price                      0.779357
total_freight                    0.779357
avg_review_score                 0.772317
lat                              0.279563
lng                              0.279563
dtype: float64

In [8]:
clean_df_orders=df_orders.copy()
clean_df_orders=clean_df_orders.dropna()

In [9]:
clean_df_orders.duplicated().sum()

np.int64(0)

### **Feature Engineering**

In [10]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    clean_df_orders[col] = pd.to_datetime(clean_df_orders[col])


clean_df_orders['purchase_year'] = clean_df_orders['order_purchase_timestamp'].dt.year
clean_df_orders['purchase_month'] = clean_df_orders['order_purchase_timestamp'].dt.month
clean_df_orders['purchase_day'] = clean_df_orders['order_purchase_timestamp'].dt.day
clean_df_orders['purchase_hour'] = clean_df_orders['order_purchase_timestamp'].dt.hour
clean_df_orders['purchase_day_name'] = clean_df_orders['order_purchase_timestamp'].dt.day_name()


clean_df_orders['season'] = clean_df_orders['purchase_month'].map({
    12:'Winter',1:'Winter',2:'Winter',
    3:'Spring',4:'Spring',5:'Spring',
    6:'Summer',7:'Summer',8:'Summer',
    9:'Autumn',10:'Autumn',11:'Autumn'
})

clean_df_orders['approval_days'] = (
    clean_df_orders['order_approved_at']
    - clean_df_orders['order_purchase_timestamp']
).dt.total_seconds() / 3600

clean_df_orders['carrier_handling_days'] = (
    clean_df_orders['order_delivered_carrier_date']
    - clean_df_orders['order_approved_at']
).dt.days

clean_df_orders['delivery_diff'] = (
    clean_df_orders['order_delivered_customer_date']
    - clean_df_orders['order_estimated_delivery_date']
).dt.days

clean_df_orders['review_category'] = pd.cut(
    clean_df_orders['avg_review_score'],
    bins=[0,2,3,4,5],
    labels=['Bad','Okay','Good','Excellent']
)

### **EDA**

#### Annual Sales and Costs Analysis

In [11]:
fig =make_subplots(
    rows=1,cols=4,subplot_titles=("total price Distribution","total freight Distribution","total paid Distribution","total number of items Distribution")
)

fig.add_trace(
    go.Box(
        y=clean_df_orders["total_price"],
        name= "total price Distribution",
        showlegend=False
    ),
    row=1,col=1
)

fig.add_trace(
    go.Box(
        y=clean_df_orders["total_freight"],
        name= "total freight Distribution",
        showlegend=False
    ),
    row=1,col=2
)

fig.add_trace(
    go.Box(
        y=clean_df_orders["total_paid"],
        name= "total paid Distribution",
        showlegend=False
    ),
    row=1,col=3
)

fig.add_trace(
    go.Box(
        y=clean_df_orders["total_items"],
        name= "total items Distribution",
        showlegend=False
    ),
    row=1,col=4
)

fig.update_layout(
    title="total price , total freight ,total paid and items Distribution",
    bargap=0.05,
)
fig.show()


> - The entire dataset is heavily driven by low value single item entries while any high priced transactions or multi item records are just rare outliers

In [12]:
clean_df_orders["purchase_year"].value_counts()

purchase_year
2018    52314
2017    42969
2016      268
Name: count, dtype: int64

> - Data from 2016 was excluded from the trend analysis due to an insufficient sample size only 268 records compared to later years making any direct comparison distorted and unfair

In [13]:
yearly_sales = clean_df_orders.groupby('purchase_year')[['total_freight', 'total_paid',"total_price"]].sum().reset_index()
yearly_sales = yearly_sales[yearly_sales['purchase_year'].isin([2017, 2018])]
yearly_sales['freight_ratio'] = (yearly_sales['total_freight']/ yearly_sales['total_paid']) * 100

In [14]:
fig = make_subplots(rows=1,cols=2,subplot_titles=( "Sales & Freight Trends Over Years","Freight Cost Percentage",))


fig.add_trace(
    go.Bar(
        x=yearly_sales['purchase_year'],
        y=yearly_sales['total_paid'],
        name='Total Paid'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=yearly_sales['purchase_year'],
        y=yearly_sales['total_price'],
        name='Total Price'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=yearly_sales['purchase_year'],
        y=yearly_sales['total_freight'],
        name='Total Freight'
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x=yearly_sales['purchase_year'],
        y=yearly_sales['freight_ratio'],
        name='Freight Ratio %',
        marker_color='green'
    ),
    row=1, col=2
)



fig.update_xaxes(type='category')
fig.update_layout(
    title='Sales and Freight Analysis Across Years',
)

fig.update_xaxes(title_text='Year', row=1, col=1)
fig.update_yaxes(title_text='Amount', row=1, col=1)

fig.update_xaxes(title_text='Year', row=1, col=2)
fig.update_yaxes(title_text='Freight Ratio (%)', row=1, col=2)

fig.show()

> - Total sales showed a clear increase from 2017 to 2018 while the ratio of freight costs remained completely flat and predictable

In [15]:
fig = px.scatter(
    clean_df_orders,
    x='total_items',
    y='total_freight',
    trendline='ols',
    title='Relationship between Items Count and Freight Cost'
)

fig.show()

> - There is no strong correlation between the number of items and freight costs as most orders stay low cost regardless of item count leaving a few extreme outliers to pull the trend line upward

#### Cities And States Analysis

In [16]:
sales_by_cities=clean_df_orders.groupby("customer_city",as_index=False)["total_paid"].sum().sort_values(by="total_paid",ascending=False)
sales_by_states= clean_df_orders.groupby("customer_state",as_index=False)["total_paid"].sum().sort_values(by="total_paid",ascending=False)

In [17]:
fig = make_subplots(rows=1,cols=2,subplot_titles=( "Top 10 Cities by Total Sales", "Top 10 States by Total Sales",
    )
)


fig.add_trace(
    go.Bar(
        x=sales_by_cities["customer_city"].head(10),
        y=sales_by_cities["total_paid"].head(10),
        name="Top Cities"
    ),
    row=1,
    col=1
)



fig.add_trace(
    go.Bar(
        x=sales_by_states["customer_state"].head(10),
        y=sales_by_states["total_paid"].head(10),
        name="Top States"
    ),
    row=1,
    col=2
)



fig.update_layout(
    title="Sales Distribution Across Cities and States",
    showlegend=False,


)


fig.update_xaxes(title_text="City", row=1, col=1)
fig.update_yaxes(title_text="Total Sales", row=1, col=1)

fig.update_xaxes(title_text="State", row=1, col=2)
fig.update_yaxes(title_text="Total Sales", row=1, col=2)

fig.show()

> - Sao Paulo (SP) completely dominates the market driving more sales individually than the next top 3 regions Rio de Janeiro  Minas Gerais and Rio Grande do Sul combined

#### Order Logistics Time Analysis

In [18]:
fig = make_subplots(rows=1,cols=3,subplot_titles=("Approval Time Distribution","Carrier Handling Time Distribution",
        "Delivery Difference Distribution"
    )
)


fig.add_trace(
    go.Histogram(
        x=clean_df_orders['approval_days'],
        nbinsx=15,
        name='Approval Time'
    ),
    row=1, col=1
)


fig.add_trace(
    go.Histogram(
        x=clean_df_orders['carrier_handling_days'],
        nbinsx=15,
        name='Carrier Handling'
    ),
    row=1, col=2
)


fig.add_trace(
    go.Histogram(
        x=clean_df_orders['delivery_diff'],
        nbinsx=15,
        name='Delivery Difference'
    ),
    row=1, col=3
)

fig.update_layout(
    title='Order Logistics Time Analysis',
    showlegend=False,
    bargap=0.05
)
fig.update_xaxes(title_text='Hours', row=1, col=1)
fig.update_yaxes(title_text='Orders Count', row=1, col=1)

fig.update_xaxes(title_text='Days', row=1, col=2)
fig.update_yaxes(title_text='Orders Count', row=1, col=2)

fig.update_xaxes(title_text='Days Difference', row=1, col=3)
fig.update_yaxes(title_text='Orders Count', row=1, col=3)

fig.show()

> - Orders get approved almost instantly in the first hour also the warehouse takes only a few days to prepare the items and hand them over to the shipping company with no delays
> - The massive concentration of orders in the negative zone of "Delivery Difference" proves that the platform consistently delivers packages ahead of schedule 

#### reviews Analysis

In [19]:
review_dist = clean_df_orders['review_category'].value_counts().reset_index()

In [20]:
fig = px.bar(
    review_dist,
    x='review_category',
    y='count',
    color='review_category',
    title='Customer Review Distribution'
)

fig.show()

> - The vast majority of customers are highly satisfied with "Excellent" ratings completely dominating the chart showing that the website is doing a great job overall

In [21]:
order_status = clean_df_orders['order_status'].value_counts().reset_index()

#### Sales by Day and Season

In [22]:
sales_by_day=clean_df_orders.groupby("purchase_day_name",as_index=False)["total_paid"].sum().sort_values(by="total_paid",ascending=False)
sales_by_seasons= clean_df_orders.groupby("season",as_index=False)["total_paid"].sum().sort_values(by="total_paid",ascending=False)
sales_by_hours= clean_df_orders.groupby("purchase_hour",as_index=False)["total_paid"].sum().sort_values(by="total_paid",ascending=False)

In [23]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Sales by Day", "Sales by Season", "Sales by Hour")
)


fig.add_trace(
    go.Bar(
        x=sales_by_day['purchase_day_name'],
        y=sales_by_day['total_paid'],
        name='Day Sales'
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x=sales_by_seasons['season'],
        y=sales_by_seasons['total_paid'],
        name='Season Sales'
    ),
    row=1, col=2
)


fig.add_trace(
    go.Bar(
        x=sales_by_hours['purchase_hour'],
        y=sales_by_hours['total_paid'],
        name='Hour Sales'
    ),
    row=1, col=3
)


fig.update_layout(
    title='Sales Distribution by Time Factors',
    showlegend=False,
)

fig.show()

> -  Most buying activity happens during weekdays and peaks during the day between 10 AM and 10 PM especially around Spring and Summer while sales completely dry up after midnight.

## **products Analysis**

In [24]:
df_products_analysis.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,order_purchase_timestamp,order_status
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,...,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,2017-09-13 08:59:02,delivered
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,...,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,2017-04-26 10:53:06,delivered
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,...,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,2018-01-14 14:33:31,delivered
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,...,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,2018-08-08 10:00:35,delivered
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,...,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,2017-02-04 13:57:51,delivered


In [25]:
df_products_analysis.describe().T

,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68
product_name_lenght,111047.0,48.775978,10.025581,5.00,42.00,52.00,57.00,76.00
product_description_lenght,111047.0,787.867029,652.135608,4.00,348.00,603.00,987.00,3992.00
product_photos_qty,111047.0,2.209713,1.721438,1.00,1.00,1.00,3.00,20.00
product_weight_g,112632.0,2093.672047,3751.596884,0.00,300.00,700.00,1800.00,40425.00
product_length_cm,112632.0,30.153669,16.153449,7.00,18.00,25.00,38.00,105.00
product_height_cm,112632.0,16.593766,13.443483,2.00,8.00,13.00,20.00,105.00
product_width_cm,112632.0,22.996546,11.707268,6.00,15.00,20.00,30.00,118.00


In [26]:
df_products_analysis.describe(include="O").T

,count,unique,top,freq
order_id,112650,98666,8272b63d03f5f79c56e9e4120aec44ef,21
product_id,112650,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527
seller_id,112650,3095,6560211a19b47992c3666cc44a7e94c0,2033
shipping_limit_date,112650,93318,2017-07-21 18:25:23,21
product_category_name,111047,73,cama_mesa_banho,11115
product_category_name_english,111023,71,bed_bath_table,11115
seller_city,112650,611,sao paulo,27983
seller_state,112650,23,SP,80342
order_purchase_timestamp,112650,98112,2017-07-16 18:19:25,21
order_status,112650,7,delivered,110197


In [27]:
df_products_analysis.isna().sum()/len(df_products_analysis)*100

order_id                         0.000000
order_item_id                    0.000000
product_id                       0.000000
seller_id                        0.000000
shipping_limit_date              0.000000
price                            0.000000
freight_value                    0.000000
product_category_name            1.422992
product_name_lenght              1.422992
product_description_lenght       1.422992
product_photos_qty               1.422992
product_weight_g                 0.015979
product_length_cm                0.015979
product_height_cm                0.015979
product_width_cm                 0.015979
product_category_name_english    1.444296
seller_zip_code_prefix           0.000000
seller_city                      0.000000
seller_state                     0.000000
order_purchase_timestamp         0.000000
order_status                     0.000000
dtype: float64

### **Missing Values Handling**

In [28]:
clean_DFproducts =df_products_analysis.copy()
clean_DFproducts=clean_DFproducts.dropna()
clean_DFproducts.duplicated().sum()

np.int64(0)

### **Feature Engineering** 

In [29]:
clean_DFproducts['product_volume'] = (
    clean_DFproducts['product_length_cm'] *
    clean_DFproducts['product_height_cm'] *
    clean_DFproducts['product_width_cm']
)

### **EDA**

#### Price and Freight value Analysis

In [30]:
fig = make_subplots(
    rows=1,
    cols=2, subplot_titles=(  "Product Price Distribution", "Freight Value Distribution",
    )
)


fig.add_trace(
    go.Histogram(
        x=clean_DFproducts["price"],
        nbinsx=15,
        name="Price"
    ),
    row=1, col=1
)


fig.add_trace(
    go.Histogram(
        x=clean_DFproducts["freight_value"],
        nbinsx=15,
        name="Freight"
    ),
    row=1, col=2
)




fig.update_layout(
    title="Product Price and Freight Cost",
    showlegend=False,
    bargap=0.05
)

fig.show()

 > - The vast majority of products are low priced under 500 with cheap shipping fees under 50 showing that the platform mostly relies on budget friendly  high volume sales

In [31]:
fig = px.scatter(
    clean_DFproducts,
    x="product_weight_g",
    y="freight_value",
    size="product_volume",
    trendline="ols",
    opacity=0.05,
    title="Effect of Weight & Volume on Freight Value"
)

fig.show()

 >- There is a clear positive correlation where heavier and bulkier products directly lead to higher shipping fees with a noticeable spike in costs for items reaching the 30kg mark

In [32]:
clean_DFproducts['customer_state'] = clean_DFproducts['order_id'].map(clean_df_orders.set_index('order_id')['customer_state'])
local_shipping = clean_DFproducts[clean_DFproducts['seller_state'] == clean_DFproducts['customer_state']]

In [33]:
total_local = local_shipping['freight_value'].sum()
total_non_local = clean_DFproducts[clean_DFproducts['seller_state'] != clean_DFproducts['customer_state']]['freight_value'].sum()

fig = px.bar(
    x=["Local Shipping", "Non-Local Shipping"],
    y=[total_local, total_non_local],
    title="Freight Cost Comparison: Local vs Non-Local"
)

fig.update_layout(
    xaxis_title="Shipping Type",
    yaxis_title="Total Freight Value",
    showlegend=False
)

fig.show()


> - The total spending on non local shipping is more than 3 times higher than local shipping passing 1.5M which shows that most delivery expenses come from long-distance orders

#### Most and Least Sold Categories

In [34]:
products =clean_DFproducts["product_category_name_english"].value_counts().sort_values(ascending=False).reset_index()

In [35]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Top 10 Product Categories",
        "Bottom 10 Product Categories"
    )
)


fig.add_trace(
    go.Bar(
        x=products["product_category_name_english"].head(10),
        y=products["count"].head(10),
        name="Top Categories"
    ),
    row=1,
    col=1
)


fig.add_trace(
    go.Bar(
        x=products["product_category_name_english"].tail(10),
        y=products["count"].tail(10),
        name="Bottom Categories"
    ),
    row=1,
    col=2
)


fig.update_layout(
    title="Product Category Distribution",
    showlegend=False,   
)


fig.update_xaxes(title_text="Category", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

fig.update_xaxes(title_text="Category", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.show()

> -  Home essentials bed_bath_table and health_beauty are the top selling categories with around 10k orders each while niche categories like security_and_services are nearly invisible with almost zero sales

#### Seller Revenue and Geographic Distribution

In [36]:
top_sellers=clean_DFproducts.groupby('seller_id')['price'].sum().sort_values(ascending=False)
top_seller_city=clean_DFproducts.groupby("seller_city")["price"].sum().sort_values(ascending=False)
top_seller_state=clean_DFproducts.groupby("seller_state")["price"].sum().sort_values(ascending=False)
city_category = clean_DFproducts.groupby(['seller_city', 'product_category_name_english']).size().reset_index(name='count')

In [37]:
fig = make_subplots( rows=2, cols=2,subplot_titles=("Top Sellers by Revenue","Top Seller Cities","Top Seller States",
        "City vs Product Category"
    ),
    horizontal_spacing=0.2 
)


fig.add_trace(
    go.Bar(
        x=top_sellers.head(10).values,
        y=top_sellers.head(10).index,
        orientation='h',
        name="Top Sellers"
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x=top_seller_city.head(10).values,
        y=top_seller_city.head(10).index,
        orientation='h',
        name="Cities"
    ),
    row=1, col=2
)


fig.add_trace(
    go.Bar(
        x=top_seller_state.head(10).values,
        y=top_seller_state.head(10).index,
        orientation='h',
        name="States"
    ),
    row=2, col=1
)


top_city_cat = city_category.sort_values('count', ascending=False).head(10)

fig.add_trace(
    go.Bar(
        x=top_city_cat['count'],
        y=top_city_cat['seller_city'] + " - " + top_city_cat['product_category_name_english'],
        orientation='h',
        name="City-Category"
    ),
    row=2, col=2
)


fig.update_layout(
    title="Seller Analysis",
    height=800,
    width=1200,
    showlegend=False
)

fig.show()

> - Sao Paulo is the absolute powerhouse for sales across cities and states

#### Order Status

In [38]:
order_status=clean_DFproducts["order_status"].value_counts().sort_values(ascending=False).reset_index()

In [39]:
fig = px.bar(
    order_status,
    x="order_status",
    y="count",
    log_y=True,
    title="Order Status Distribution"
)

fig.update_layout(
    xaxis_title="Order Status",
    yaxis_title="Orders Count (Log Scale)",
    showlegend=False
)

fig.show()

> - Over 108k products were successfully delivered while 1158 products are currently on the way shipped meaning almost all items ordered reach the customers safely

## **Payments Analysis**

In [40]:
df_payments_analysis.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_purchase_timestamp,customer_id,customer_state,customer_city
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2018-04-25 22:01:49,0a8556ac6be836b46b3e89920d59291c,MG,teofilo otoni
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2018-06-26 11:01:38,f2c7fc58a9de810828715166c672f10a,SP,sao paulo
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2017-12-12 11:19:55,25b14b69de0b6e184ae6fe2755e478f9,SP,sao paulo
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2017-12-06 12:04:06,7a5d8efaaa1081f800628c30d2b0728f,MG,juiz de fora
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2018-05-21 13:59:17,15fd6fb8f8312dbb4674e4518d6fa3b3,SP,conchas


In [41]:
df_payments_analysis.describe().T

,count,mean,std,min,25%,50%,75%,max
payment_sequential,103886.0,1.092679,0.706584,1.0,1.00,1.0,1.0000,29.00
payment_installments,103886.0,2.853349,2.687051,0.0,1.00,1.0,4.0000,24.00
payment_value,103886.0,154.100380,217.494064,0.0,56.79,100.0,171.8375,13664.08


In [42]:
df_payments_analysis.describe(include="O").T

,count,unique,top,freq
order_id,103886,99440,fa65dad1b0e818e3ccc5cb0e39231352,29
payment_type,103886,5,credit_card,76795
order_purchase_timestamp,103886,98874,2017-04-20 12:45:34,29
customer_id,103886,99440,9af2372a1e49340278e7c1ef8d749f34,29
customer_state,103886,27,SP,43622
customer_city,103886,4119,sao paulo,16221


In [43]:
df_payments_analysis.isna().sum()

order_id                    0
payment_sequential          0
payment_type                0
payment_installments        0
payment_value               0
order_purchase_timestamp    0
customer_id                 0
customer_state              0
customer_city               0
dtype: int64

In [44]:
df_payments =df_payments_analysis.copy()

### **EDA**

#### Number of Payment Installments

In [45]:
payment_sequential = df_payments["payment_sequential"].value_counts(ascending=False).reset_index()
payment_type = df_payments["payment_type"].value_counts(ascending=False).reset_index()
payment_installments = df_payments["payment_installments"].value_counts(ascending=False).reset_index()

In [46]:
fig = make_subplots(rows=1,cols=3,subplot_titles=( "Payment Sequential Distribution","Payment Type Distribution",
        "Installments Distribution"
    )
)


fig.add_trace(
    go.Bar(
        x=payment_sequential['payment_sequential'],
        y=payment_sequential['count'],
        name='Sequential'
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x=payment_type['payment_type'],
        y=payment_type['count'],
        name='Payment Type'
    ),
    row=1, col=2
)


fig.add_trace(
    go.Bar(
        x=payment_installments['payment_installments'],
        y=payment_installments['count'],
        name='Installments'
    ),
    row=1, col=3
)

fig.update_layout(
    title="Payment Behavior Analysis",
    showlegend=False,
    bargap=0.2
)

fig.update_yaxes(type='log')

fig.update_xaxes(title_text="Payment Sequential", row=1, col=1)
fig.update_yaxes(title_text="Count (Log Scale)", row=1, col=1)

fig.update_xaxes(title_text="Payment Type", row=1, col=2)
fig.update_yaxes(title_text="Count (Log Scale)", row=1, col=2)

fig.update_xaxes(title_text="Installments", row=1, col=3)
fig.update_yaxes(title_text="Count (Log Scale)", row=1, col=3)

fig.show()

>- Most customers prefer paying with credit cards and usually clear their total in a single payment, while a significant portion uses boleto or splits costs into smaller installments.

#### VIP Customer Analysis 

In [47]:
top_customers = df_payments.groupby('customer_id',as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0])
).sort_values(by='payment_value', ascending=False).head(10)

top_10_vip_ids = top_customers["customer_id"].tolist()

vip_orders_fixed = df_orders[df_orders['customer_id'].isin(top_10_vip_ids)]

In [48]:
vip_customer_id = vip_orders_fixed.groupby('customer_id').agg(customer_unique_id=('customer_unique_id', lambda x: x.unique())).reset_index()
vip_customer_id['customer_unique_id'] = vip_customer_id['customer_unique_id'].apply(lambda x: x[0])
vip_customer_id['customer_short'] = vip_customer_id['customer_unique_id'].apply(lambda x: x[-8:])

In [49]:
df_products_with_customers = clean_DFproducts.merge(
    df_orders[['order_id', 'customer_id']], 
    on='order_id', 
    how='left'
)
vip_products_final = df_products_with_customers[df_products_with_customers['customer_id'].isin(top_10_vip_ids)]

vip_customer_products = vip_products_final.groupby('customer_id').agg(
    products=('product_category_name_english', lambda x: list(x.unique())),
    total_items=('product_id', 'count')
).reset_index()

vip_customer_products['products'] = vip_customer_products['products'].apply(lambda x: x[0])

In [50]:
top_customers = top_customers.merge(
    vip_customer_id[['customer_id', 'customer_short']],
    on='customer_id',
    how='left'
)

vip_customer_products=vip_customer_products.merge(
    vip_customer_id[['customer_id', 'customer_short']],
    on='customer_id',
    how='left'
)

In [51]:
payment_dist = top_customers['payment_type'].value_counts().reset_index()
payment_dist.columns = ['payment_type', 'count']

fig = make_subplots(rows=1,cols=3,specs=[ [{"type": "bar"}, {"type": "pie"}, {"type": "bar"}]],
    subplot_titles=(
        "Top Customers by Payment Value",
        "Payment Type Distribution",
        "Products Count per Customer"
    )
)

fig.add_trace(
    go.Bar(
        x=top_customers['customer_short'],
        y=top_customers['payment_value'],
        name='Top Customers',
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=payment_dist['payment_type'],
        values=payment_dist['count'],
        hole=0.4
    ),
    row=1, col=2
)


for prod in vip_customer_products['products'].unique():

    temp = vip_customer_products[
        vip_customer_products['products'] == prod
    ]

    fig.add_trace(
        go.Bar(
            x=temp['customer_short'],
            y=temp['total_items'],
            name=prod
        ),row=1, col=3
    )



fig.update_layout(
    title='VIP Customer Analysis Dashboard',
   
)

fig.show()

> - The top VIP customer leads with a massive 14k payment while credit cards remain the preferred payment type 60% for these premium orders

## **summary**

> - The vast majority of items over 108k are successfully delivered
> - Switching to a log scale revealed that around 1158 items are currently shipped while cancellations are very low
> - Non-local shipping is a massive expense costing over 1.5M which is more than 3 times higher than local shipping
> - Sao Paulo completely dominates the platform as both the top selling state and city.
> - The single highest performing segment on the platform is ibitinga - bed_bath_table.
> - Most customers prefer using credit cards followed by boleto invoices
> - While many buyers pay upfront in one go a huge chunk of customers rely heavily on splitting their costs into multiple installments
> - A small group of VIP clients drives massive revenue the top VIP customer spent 14k on a bulk order of 8 telecom items
> - 60% of these big spenders use credit cards but 40% rely on boletos which means the business needs to make sure invoice collection is fast